# Chuyển `customer_turns` thành audio bằng F5-TTS-Vietnamese

Notebook Google Colab này đọc **một file scenario JSON**, lấy `calls.<call_id>.customer_turns`, rồi tạo **một file WAV cho mỗi cuộc gọi** bằng checkpoint `hynt/F5-TTS-Vietnamese-ViVoice`. Model chạy cục bộ trên GPU Colab, không cần API key.

Trước khi chạy:

1. Chọn **Runtime → Change runtime type → T4 GPU** (hoặc GPU mạnh hơn).
2. Đặt file giọng mẫu trong Google Drive và nhập đúng `REF_AUDIO`, `REF_TEXT`. Nếu đường dẫn `REF_AUDIO` không tồn tại, notebook sẽ mở hộp để bạn upload file trực tiếp. Nên dùng audio sạch, một người nói, khoảng 3–10 giây.
3. Chọn `SCENARIO_FILE`, kiểm tra phần xem trước, rồi đổi `RUN_TTS = True`.

> `RUN_TTS = False` theo mặc định để Run All không tự tải checkpoint lớn hoặc bắt đầu suy luận. Model có giấy phép **CC-BY-NC-SA-4.0 (phi thương mại)**. Chỉ clone giọng khi có sự đồng ý hợp pháp và nên công khai rằng audio được tạo bởi AI.

Nguồn: [model card](https://huggingface.co/hynt/F5-TTS-Vietnamese-ViVoice) · [repository](https://github.com/nguyenthienhy/F5-TTS-Vietnamese)

## 0. Cài đặt F5-TTS-Vietnamese

Ô này clone repository và cài thư viện. Có thể mất vài phút. Fork Vietnamese cũ ghim một bản NumPy không tương thích Python 3.13, vì vậy notebook kiểm tra NumPy trước và không cho dependency cũ hạ phiên bản. Nếu ô này báo đã sửa NumPy, hãy restart session rồi chạy lại notebook từ đầu.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

F5_REPO_URL = "https://github.com/nguyenthienhy/F5-TTS-Vietnamese.git"
F5_REPO_DIR = Path("/content/F5-TTS-Vietnamese")

if not F5_REPO_DIR.is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", F5_REPO_URL, str(F5_REPO_DIR)],
        check=True,
    )
else:
    print(f"Dùng lại repository đã có: {F5_REPO_DIR}")

# Kiểm tra NumPy trong một process sạch để phát hiện cài đặt bị trộn/hỏng.
numpy_probe = subprocess.run(
    [
        sys.executable,
        "-c",
        "import numpy; import numpy.strings; print(numpy.__version__)",
    ],
    capture_output=True,
    text=True,
)
if numpy_probe.returncode != 0:
    print("NumPy hiện tại bị lỗi; đang cài NumPy 2.2.6 cho Python 3.13...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--force-reinstall",
            "--no-cache-dir",
            "numpy==2.2.6",
        ],
        check=True,
    )
    raise RuntimeError(
        "Đã sửa NumPy. Chọn Runtime → Restart session, sau đó chạy lại notebook từ đầu."
    )

print("NumPy process sạch:", numpy_probe.stdout.strip())

# Lấy dependency tương thích Python mới từ F5-TTS hiện hành.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "f5-tts==1.1.22",
        "jieba",
    ],
    check=True,
)

# Dùng code Vietnamese nhưng bỏ resolver dependency cũ (đặc biệt là NumPy<=1.26.4).
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "-e",
        str(F5_REPO_DIR),
    ],
    check=True,
)

# Editable install đôi khi chưa được kernel Colab nhận ngay; thêm src trực tiếp.
F5_SRC_DIR = F5_REPO_DIR / "src"
if str(F5_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(F5_SRC_DIR))

if shutil.which("ffmpeg") is None:
    raise EnvironmentError(
        "Không tìm thấy ffmpeg. Hãy chạy trong một cell: !apt-get update -qq && apt-get install -y -qq ffmpeg"
    )

from f5_tts.api import F5TTS

print("Đã cài và import thành công F5-TTS-Vietnamese.")

In [ ]:
import gc
import json
import re
from pathlib import Path

import torch
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display

print("PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Mount Google Drive và chọn dữ liệu

`REF_TEXT` phải là transcript **chính xác** của `REF_AUDIO`. Không nên để trống vì repository sẽ tải thêm mô hình ASR và có thể nhận dạng tiếng Việt sai. Nếu `REF_AUDIO` không tồn tại, Colab sẽ yêu cầu upload một file audio. `OUTPUT_DIR = Path("/content")` là bộ nhớ tạm; đổi sang một thư mục Drive nếu muốn giữ kết quả sau khi Colab ngắt phiên.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DATASET_DIR = Path("/content/drive/MyDrive/scenarios_complete dataset")
SCENARIO_FILE = DATASET_DIR / "SC-001.json"

# Thay bằng file giọng mẫu của bạn và transcript đúng từng chữ.
REF_AUDIO = Path("/content/drive/MyDrive/f5-tts/reference.wav")
REF_TEXT = "Nhập transcript chính xác của file giọng mẫu tại đây."

OUTPUT_DIR = Path("/content")
MODEL_CACHE_DIR = Path("/content/f5-tts-vietnamese-model")

if not SCENARIO_FILE.is_file():
    raise FileNotFoundError(f"Không tìm thấy file scenario: {SCENARIO_FILE}")

print("Input       :", SCENARIO_FILE)
if REF_AUDIO.is_file():
    print("Reference   :", REF_AUDIO)
else:
    print("Reference   : chưa tìm thấy; Colab sẽ yêu cầu upload khi chạy TTS")
print("Output root :", OUTPUT_DIR)

## 2. Đọc và kiểm tra `customer_turns`

In [ ]:
def load_customer_turns(json_path: Path) -> tuple[str, dict[str, list[str]]]:
    """Đọc scenario và trả về (scenario_id, {call_id: customer_turns})."""
    with json_path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    if not isinstance(payload, dict):
        raise ValueError("Nội dung JSON phải là một object.")

    scenario_id = str(payload.get("scenario_id") or json_path.stem)
    calls = payload.get("calls")
    if not isinstance(calls, dict) or not calls:
        raise ValueError("Không tìm thấy object 'calls' hoặc 'calls' đang rỗng.")

    turns_by_call: dict[str, list[str]] = {}
    for call_id, call_data in calls.items():
        if not isinstance(call_data, dict):
            raise ValueError(f"{call_id} phải là một object.")

        turns = call_data.get("customer_turns")
        if not isinstance(turns, list):
            raise ValueError(f"{call_id}.customer_turns phải là một list.")
        if not all(isinstance(turn, str) for turn in turns):
            raise ValueError(f"Mọi phần tử trong {call_id}.customer_turns phải là chuỗi.")

        cleaned_turns = [turn.strip() for turn in turns if turn.strip()]
        if not cleaned_turns:
            raise ValueError(f"{call_id}.customer_turns không có câu hợp lệ.")
        turns_by_call[str(call_id)] = cleaned_turns

    return scenario_id, turns_by_call


scenario_id, customer_turns_by_call = load_customer_turns(SCENARIO_FILE)
print(f"Scenario: {scenario_id}")
for call_id, turns in customer_turns_by_call.items():
    print(f"\n{call_id} ({len(turns)} câu):")
    for index, text in enumerate(turns, start=1):
        print(f"  {index}. {text}")

## 3. Cấu hình suy luận

- `CALL_IDS = None`: xử lý tất cả call; ví dụ `['call_1']` để chạy thử một call.
- `SPEED`: tốc độ đọc; `1.0` là mặc định.
- `NFE_STEP = 32`: chất lượng mặc định; giảm xuống 16 để thử nhanh hơn.
- `OVERWRITE = False`: bỏ qua WAV đã tồn tại.
- Model và vocoder chỉ được tải khi `RUN_TTS = True`.

In [ ]:
MODEL_REPO_ID = "hynt/F5-TTS-Vietnamese-ViVoice"
MODEL_NAME = "F5TTS_Base"
SPEED = 1.0
NFE_STEP = 32
CFG_STRENGTH = 2.0
SWAY_SAMPLING_COEF = -1.0

CALL_IDS = None  # None = tất cả; ví dụ: ["call_1"]
OVERWRITE = False
RUN_TTS = False  # Đổi thành True sau khi kiểm tra dữ liệu và reference.

if SPEED <= 0:
    raise ValueError("SPEED phải lớn hơn 0.")
if NFE_STEP <= 0:
    raise ValueError("NFE_STEP phải lớn hơn 0.")

In [ ]:
def safe_filename(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("._") or "audio"


def resolve_reference_audio(configured_path: Path) -> Path:
    """Dùng file Drive nếu có; nếu thiếu thì yêu cầu upload trong Colab."""
    configured_path = Path(configured_path)
    if configured_path.is_file():
        return configured_path

    print(f"Không tìm thấy REF_AUDIO: {configured_path}")
    print("Hãy chọn đúng một file giọng mẫu (WAV, MP3, M4A...).")
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            f"Không tìm thấy file giọng mẫu: {configured_path}"
        ) from exc

    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("Bạn chưa upload file giọng mẫu.")
    if len(uploaded) != 1:
        raise ValueError("Chỉ upload đúng một file giọng mẫu.")

    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    upload_dir = Path("/content/f5-tts-reference")
    upload_dir.mkdir(parents=True, exist_ok=True)
    uploaded_path = upload_dir / Path(uploaded_name).name
    uploaded_path.write_bytes(uploaded_bytes)
    print("Đã nhận file giọng mẫu:", uploaded_path)
    return uploaded_path


def build_call_text(text_chunks: list[str]) -> str:
    """Ghép các lượt khách; thêm dấu chấm khi một lượt chưa có dấu kết câu."""
    sentences = []
    for chunk in text_chunks:
        sentence = chunk.strip()
        if not sentence:
            continue
        if sentence[-1] not in ".!?;:":
            sentence += "."
        sentences.append(sentence)

    if not sentences:
        raise ValueError("Không có nội dung hợp lệ để tạo audio.")
    return "\n".join(sentences)


def select_call_ids(
    turns_by_call: dict[str, list[str]],
    requested_call_ids: list[str] | None,
) -> list[str]:
    if requested_call_ids is None:
        return list(turns_by_call)
    if isinstance(requested_call_ids, str):
        raise TypeError("CALL_IDS phải là list, ví dụ ['call_1'], không phải chuỗi.")

    selected = list(dict.fromkeys(requested_call_ids))
    missing = [call_id for call_id in selected if call_id not in turns_by_call]
    if missing:
        raise KeyError(f"CALL_IDS không tồn tại trong file: {missing}")
    return selected


def download_model_files(model_dir: Path) -> tuple[Path, Path]:
    """Tải đúng checkpoint và vocabulary cần cho inference."""
    model_dir.mkdir(parents=True, exist_ok=True)
    ckpt_file = Path(
        hf_hub_download(
            repo_id=MODEL_REPO_ID,
            filename="model_last.pt",
            local_dir=model_dir,
        )
    )
    vocab_file = Path(
        hf_hub_download(
            repo_id=MODEL_REPO_ID,
            filename="config.json",
            local_dir=model_dir,
        )
    )
    if not ckpt_file.is_file() or not vocab_file.is_file():
        raise FileNotFoundError("Tải model chưa đầy đủ: cần model_last.pt và config.json.")
    return ckpt_file, vocab_file


def import_f5tts_class():
    """Bảo đảm source/package F5-TTS có thể import trong kernel hiện tại."""
    import importlib
    import subprocess
    import sys

    repo_dir = Path("/content/F5-TTS-Vietnamese")
    src_dir = repo_dir / "src"

    if not src_dir.is_dir():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/nguyenthienhy/F5-TTS-Vietnamese.git",
                str(repo_dir),
            ],
            check=True,
        )

    if str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))

    try:
        from f5_tts.api import F5TTS
    except ModuleNotFoundError as exc:
        if exc.name and exc.name.startswith("numpy"):
            raise RuntimeError(
                "NumPy trong kernel đang bị lỗi. Hãy chạy lại ô cài đặt đầu tiên, "
                "restart session theo hướng dẫn, rồi chạy notebook từ đầu."
            ) from exc
        print("Package F5-TTS chưa đầy đủ; đang cài lại trong kernel hiện tại...")
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "f5-tts==1.1.22",
                "jieba",
            ],
            check=True,
        )
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-deps",
                "-e",
                str(repo_dir),
            ],
            check=True,
        )
        importlib.invalidate_caches()
        from f5_tts.api import F5TTS

    return F5TTS


def load_f5_engine(model_dir: Path):
    """Tải checkpoint vào GPU đúng một lần và trả về engine F5TTS."""
    if not torch.cuda.is_available():
        raise EnvironmentError(
            "Chưa có CUDA. Trong Colab, chọn Runtime → Change runtime type → T4 GPU."
        )

    F5TTS = import_f5tts_class()

    ckpt_file, vocab_file = download_model_files(model_dir)
    print("Checkpoint:", ckpt_file)
    print("Vocabulary:", vocab_file)
    engine = F5TTS(
        model=MODEL_NAME,
        ckpt_file=str(ckpt_file),
        vocab_file=str(vocab_file),
        device="cuda",
    )
    return engine, ckpt_file, vocab_file

## 4. Tạo WAV và manifest

Kết quả được lưu trong `<OUTPUT_DIR>/<scenario_id>/`, mỗi call có một file WAV và toàn bộ trạng thái được ghi vào `manifest_f5_tts_vietnamese.json`. Nếu một call lỗi, notebook ghi lỗi vào manifest rồi tiếp tục call kế tiếp.

In [ ]:
if not RUN_TTS:
    print("Chưa chạy TTS. Hãy đổi RUN_TTS = True trong ô cấu hình khi sẵn sàng.")
else:
    if not REF_TEXT.strip() or REF_TEXT.startswith("Nhập transcript"):
        raise ValueError("Hãy thay REF_TEXT bằng transcript chính xác của REF_AUDIO.")

    reference_audio = resolve_reference_audio(REF_AUDIO)

    selected_call_ids = select_call_ids(customer_turns_by_call, CALL_IDS)
    scenario_output_dir = OUTPUT_DIR / safe_filename(scenario_id)
    scenario_output_dir.mkdir(parents=True, exist_ok=True)

    print("Đang tải model/vocoder và nạp lên GPU...")
    engine, ckpt_file, vocab_file = load_f5_engine(MODEL_CACHE_DIR)

    results = []
    preview_files = []

    for call_id in selected_call_ids:
        output_path = scenario_output_dir / (
            f"{safe_filename(scenario_id)}_{safe_filename(call_id)}_f5tts_customer.wav"
        )
        temp_path = output_path.with_name(output_path.stem + ".partial.wav")

        if output_path.exists() and not OVERWRITE:
            print(f"Bỏ qua file đã tồn tại: {output_path.name}")
            results.append(
                {
                    "call_id": call_id,
                    "status": "skipped_existing",
                    "output_file": str(output_path),
                }
            )
            preview_files.append(output_path)
            continue

        text = build_call_text(customer_turns_by_call[call_id])
        temp_path.unlink(missing_ok=True)
        try:
            _, sample_rate, _ = engine.infer(
                ref_file=str(reference_audio),
                ref_text=REF_TEXT.strip().lower(),
                gen_text=text,
                speed=SPEED,
                nfe_step=NFE_STEP,
                cfg_strength=CFG_STRENGTH,
                sway_sampling_coef=SWAY_SAMPLING_COEF,
                file_wave=str(temp_path),
            )
            if not temp_path.is_file() or temp_path.stat().st_size == 0:
                raise RuntimeError("F5-TTS không tạo được file WAV hợp lệ.")

            temp_path.replace(output_path)
            byte_count = output_path.stat().st_size
            preview_files.append(output_path)
            results.append(
                {
                    "call_id": call_id,
                    "status": "generated",
                    "turn_count": len(customer_turns_by_call[call_id]),
                    "text_characters": len(text),
                    "sample_rate": int(sample_rate),
                    "bytes": byte_count,
                    "output_file": str(output_path),
                }
            )
            print(f"Đã ghi {byte_count:,} bytes: {output_path}")
        except Exception as exc:
            results.append(
                {
                    "call_id": call_id,
                    "status": "error",
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )
            print(f"Lỗi ở {call_id}: {type(exc).__name__}: {exc}")
        finally:
            temp_path.unlink(missing_ok=True)
            gc.collect()
            torch.cuda.empty_cache()

    manifest_path = scenario_output_dir / "manifest_f5_tts_vietnamese.json"
    manifest = {
        "source_file": str(SCENARIO_FILE),
        "scenario_id": scenario_id,
        "model_repo_id": MODEL_REPO_ID,
        "model_name": MODEL_NAME,
        "checkpoint_file": str(ckpt_file),
        "vocab_file": str(vocab_file),
        "reference_audio": str(reference_audio),
        "speed": SPEED,
        "nfe_step": NFE_STEP,
        "cfg_strength": CFG_STRENGTH,
        "sway_sampling_coef": SWAY_SAMPLING_COEF,
        "results": results,
    }
    manifest_path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Manifest:", manifest_path)

    if preview_files:
        print("Nghe thử file đầu tiên:")
        display(Audio(filename=str(preview_files[0])))